In [ ]:
#lOAD PACKAGES
import pandas as pd
import matplotlib.pyplot as plt
import os, sys
import plotly.express as px
import plotly.graph_objects as go

import numpy as np
import itertools
# Import the processing module from the same folder
sys.path.append(os.path.join("..", "scripts", "analysis"))
sys.path.append(os.path.join("..", "model", "utils"))
from processing import transform_to_internal_time
from cumulative_reserve_mu import calculate_mu_for_each_imbalance, calculate_mu_from_quantiles, calculate_mu
from plotly.subplots import make_subplots
# import processing 
# from pivottablejs import pivot_ui
G_save = False

dim = (1000,500)
g_BLUE = "1616A7"
g_GREY = "#7F7F7F"
g_BLUE_SCALE = [
    "rgba(22, 22, 167, 0.08)",
    "rgba(22, 22, 167, 0.13)",
    "rgba(22, 22, 167, 0.18)"
],

legend_attr = dict(
    x=0.5,
    y=-0.25,
    yanchor="bottom",
    xanchor="center",
    orientation="h"
)


In [ ]:
solution_folder = "RTS-GMLC_v2.1"
demand = pd.read_csv(os.path.join("..", "input", solution_folder, 'uc','Demand.csv'))
demand = transform_to_internal_time(demand)
random_demand = pd.read_csv(os.path.join("..", "input", solution_folder, 'ed','random_demand.csv'))
random_demand = transform_to_internal_time(random_demand)
reserve = pd.read_csv(os.path.join("..", "input", solution_folder, 'uc','Reserve.csv'))
reserve = transform_to_internal_time(reserve)

solution_folder_2 = "RTS-GMLC_v2.4.2"
e_reserve_multiplier = pd.read_csv(os.path.join("..", "input", solution_folder_2, 'uc','configuration_envelopes_e_reserve_mu_v3.csv'), index_col=['day', 'hour'])

# Benchmark: cumulative reserve $\mu$ vs e-reserve $\mu$

## Cumulative reserve  $\mu$

The cumulative reserve $\mu$ is calculated based on the following metric,

$$
\mu_{t, \sigma}^{+} = \frac{\sum_{\tau=1}^{t} X_{\tau,\sigma}^{+}}{\sum_{\tau=1}^{t} R_\tau^{\uparrow}},
$$

where $X_{t,\sigma}^{+}$ corresponds to the positive imbalance taking at time step $t \in \{1, \ldots, 24\}$, for a given imbalance scenario $\sigma \in \Sigma$.

This metric measures what portion of the scheduled reserve was used during the whole operation horizon. These metrics give a proxy of the multiplier itself, as the level of reserve activation due to imbalances has a direct impact on the SOE.

We align with CAISO that uses a constant multiplier both in the up and down direction. By means of this, we assume that the selected multiplier needs to provide ensure energy for 97.5% of the cases.

NOTE: The methodology should have used $X_{\tau,\sigma}$ instead of $X_{\tau,\sigma}^{+}$

In [ ]:

mu = calculate_mu_for_each_imbalance(demand, random_demand, reserve.reset_index()
).reset_index()
mu_q = calculate_mu(demand, random_demand, reserve, quantile=0.975)
# mu_q_end = calculate_mu(demand, random_demand, reserve, quantile=0.975, last=True)

In [ ]:

day = 131
cols = ["mu_up", "mu_down"]

mu_day = mu[mu["day"] == day]
mu_q_day = mu_q[mu_q["day"] == day]

fig = make_subplots(rows=1, cols=2, subplot_titles=[c.capitalize() for c in cols])

for i, c in enumerate(cols, start=1):
    fig.add_trace(
        go.Box(x=mu_day["hour"], y=mu_day[c], name=f"{c} (box)", marker_color=g_GREY),
        row=1, col=i
    )
    fig.add_trace(
        go.Scatter(x=mu_q_day["hour"], y=mu_q_day[c], mode="lines+markers", name=f"{c} (0.975)"),
        row=1, col=i
    )

fig.update_layout(
    title=f"Multipliers for Day {day}",
    legend=legend_attr,
    width=dim[0],
    height=dim[1]
)

fig.update_xaxes(title_text="hour", row=1, col=1)
fig.update_xaxes(title_text="hour", row=1, col=2)
fig.update_yaxes(title_text="mu_up", row=1, col=1)
fig.update_yaxes(title_text="mu_down", row=1, col=2)

fig.show()

## Comparison with the e-reserve multiplier

In [ ]:
cols = ["up", "down"]

mu_q_day = mu_q[mu_q["day"] == day]
e_reserve_multiplier_ = e_reserve_multiplier.xs(day, level="day").reset_index()

fig = make_subplots(rows=1, cols=2, subplot_titles=[c.capitalize() for c in cols])

for i, c in enumerate(cols, start=1):
    fig.add_trace(
        go.Scatter(x=e_reserve_multiplier_["hour"], y=e_reserve_multiplier_[f"mu_2_{c}"], mode="lines+markers", name=f"mu_2_{c}"),
        row=1, col=i
    )
    fig.add_trace(
        go.Scatter(x=mu_q_day["hour"], y=mu_q_day[f"mu_{c}"], mode="lines+markers", name=f"mu_q {c}"),
        row=1, col=i
    )


fig.update_layout(
    title=f"mu_q vs e-reserve multiplier (Day {day})",
    legend=legend_attr,
    width=dim[0],
    height=dim[1]
)
fig.update_xaxes(title_text="hour", row=1, col=1)
fig.update_xaxes(title_text="hour", row=1, col=2)
fig.update_yaxes(title_text="value", row=1, col=1)
fig.update_yaxes(title_text="value", row=1, col=2)

fig.show()